# Mini-Project MP03 — Press Release to Plot

## Industry Comparison: Financial Services and Travel and Hospitality

*CIS 3120 — Programming for Analytics*
*Baruch College, Zicklin School of Business*

---

**Team number:** 11

**Team members:**
- Financial Services Pipeline Lead: Swikriti Kc
- Travel and Hospitality Pipeline Lead: Imran Aslam
- Comparison and Visualization Lead (Integrator): Gabriel Asimakopoulos

**Submission filename:** MP03_Notebook_team_11.ipynb

---

## How to use this starter

1. Make a copy of this notebook and rename it `MP03_Notebook_team_<NN>.ipynb` using your team number.
2. Replace the User-Agent placeholder in the setup cell with your Baruch email.
3. Configure your Anthropic API key in Colab Secrets as `ANTHROPIC_API_KEY`.
4. Work through the notebook section by section. Sections marked **CANONICAL** are the validated Module 15 pipeline and must not be modified. Sections marked **TODO** are where your team writes new code.
5. Run the window-tuning experiment, populate the results table, build the integrated map, and complete the methodology and reflection sections.
6. Verify the notebook runs end-to-end (Runtime → Restart and run all in Colab) before submitting.

See `docs/MP03_Assignment.docx` for the full assignment specification.

---

## 1. Setup

Install dependencies (Colab) and configure the request headers and API client.

In [1]:
# Colab installs (silent). The other packages are pre-installed in the Colab base image.
!pip install anthropic folium --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 763.1/763.1 kB 9.6 MB/s eta 0:00:00


In [2]:
import json
import re
import time
from datetime import date, datetime, timedelta

import requests
from bs4 import BeautifulSoup
import folium
import pandas as pd
from anthropic import Anthropic

# ─────────────────────────────────────────────────────────────────────────
# CRITICAL: Replace the placeholder below with your Baruch email.
# Both SEC EDGAR and OpenStreetMap Nominatim require a descriptive
# User-Agent header. Generic agents are rejected with HTTP 403.
# ─────────────────────────────────────────────────────────────────────────
USER_AGENT = "CIS3120 MP03 Team 11 - swikriti.kc@baruchmail.cuny.edu"

REQUEST_HEADERS = {"User-Agent": USER_AGENT}

# ─────────────────────────────────────────────────────────────────────────
# Endpoints and constants
# ─────────────────────────────────────────────────────────────────────────
EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
NOMINATIM_URL    = "https://nominatim.openstreetmap.org/search"

EDGAR_PAUSE      = 0.15   # seconds between EDGAR requests (SEC: 10 req/sec)
NOMINATIM_PAUSE  = 1.10   # seconds between Nominatim requests (1 req/sec)

# Anthropic model: current Haiku in the Claude 4.5 family.
MODEL_ID = "claude-haiku-4-5-20251001"

In [4]:
# Configure the Anthropic API client from Colab Secrets.
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
client = Anthropic(api_key=ANTHROPIC_API_KEY)

In [5]:
!git clone https://github.com/GAsimakopoulos682/cis3120-spring2026.git
%cd cis3120-spring2026

Cloning into 'cis3120-spring2026'...
remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 81 (delta 27), reused 21 (delta 21), pack-reused 50 (from 2)
Receiving objects: 100% (81/81), 117.46 KiB | 1010.00 KiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/cis3120-spring2026


In [6]:
# Import the seeded ticker lists and search-phrase lists from the mp03 module.
# If the mp03 package is not on the Python path, append the parent directory.
import sys
from pathlib import Path

# When running in Colab from a cloned repo, this places the repo root on sys.path.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mp03.seeds import (
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
)

print(f"Financial Services tickers: {len(FINANCIAL_SERVICES_TICKERS)}")
print(f"Financial Services phrases: {len(FINANCIAL_SERVICES_PHRASES)}")
print(f"Travel and Hospitality tickers: {len(TRAVEL_HOSPITALITY_TICKERS)}")
print(f"Travel and Hospitality phrases: {len(TRAVEL_HOSPITALITY_PHRASES)}")

Financial Services tickers: 14
Financial Services phrases: 40
Travel and Hospitality tickers: 14
Travel and Hospitality phrases: 40


---

## 2. Canonical Pipeline (Module 15)

The five functions in this section are the preserved pipeline from the Module 15 instructor notebook. **Do not modify these signatures.** Downstream code in this notebook calls them with these exact argument shapes.

### Stage 1 — Retrieve candidate 8-K filings from EDGAR

Each phrase is queried independently. Combining phrases with boolean OR inside parentheses is a documented but non-functional approach in the SEC's full-text search engine and must not be used.

In [7]:
def search_edgar_one_phrase(
    phrase: str,
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
) -> tuple[list[dict], int]:
    """Query EDGAR full-text search for one phrase across a date window.

    Returns a tuple of (list of hit dicts, total reported by EDGAR).
    """
    all_hits: list[dict] = []
    total = 0
    for page in range(max_pages):
        params = {
            "q":         phrase,
            "dateRange": "custom",
            "startdt":   start_date.isoformat(),
            "enddt":     end_date.isoformat(),
            "forms":     forms,
            "from":      page * 100,
        }
        response = requests.get(
            EDGAR_SEARCH_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        hits = data.get("hits", {}).get("hits", [])
        all_hits.extend(hits)
        total = data.get("hits", {}).get("total", {}).get("value", 0)
        if (page + 1) * 100 >= total:
            break
        time.sleep(EDGAR_PAUSE)
    return all_hits, total

In [8]:
def search_edgar_all_phrases(
    phrases: list[str],
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
    max_filings: int = 250,
) -> list[dict]:
    """Run search_edgar_one_phrase across a list of phrases with retry-with-backoff.

    Deduplicates by (accession number, exhibit filename). Stops accumulating
    once max_filings is reached.
    """
    seen: set[str] = set()
    deduped: list[dict] = []
    backoff_waits = [5, 10, 15]

    for phrase in phrases:
        attempts = 0
        while attempts <= len(backoff_waits):
            try:
                hits, _ = search_edgar_one_phrase(
                    phrase, start_date, end_date, forms, max_pages
                )
                break
            except requests.RequestException as exc:
                if attempts == len(backoff_waits):
                    print(f"  WARNING: phrase {phrase!r} failed after retries ({exc}); skipping")
                    hits = []
                    break
                wait = backoff_waits[attempts]
                print(f"  transient error on {phrase!r}: {exc}. retrying in {wait}s...")
                time.sleep(wait)
                attempts += 1

        for hit in hits:
            key = hit.get("_id", "")
            if key and key not in seen:
                seen.add(key)
                deduped.append(hit)
            if len(deduped) >= max_filings:
                return deduped
        time.sleep(EDGAR_PAUSE)

    return deduped

### Stage 2 — Fetch the press release text from each filing

In [9]:
def build_exhibit_url(hit: dict) -> str:
    """Construct the SEC archive URL for the exhibit referenced by the hit."""
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession_no_dashes}/{filename}"
    )


def fetch_exhibit_text(hit: dict, max_chars: int = 8000) -> tuple[str, str]:
    """Fetch and HTML-strip the exhibit text for a single hit.

    Returns (text, url). Truncates at max_chars (~2000 tokens).
    """
    url = build_exhibit_url(hit)
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    if len(text) > max_chars:
        text = text[:max_chars] + " […truncated…]"
    return text, url

### Stage 3 — Classify and extract with the Anthropic API

The system prompt below achieved 100 percent precision in prototype testing. Use it verbatim.

In [10]:
EXTRACTION_SYSTEM_PROMPT = """You are an analyst reviewing 8-K filing exhibits to identify announcements of location-related corporate events: openings, closings, relocations, or expansions of physical facilities (stores, warehouses, distribution centers, offices, plants).

Return ONLY a JSON object with these exact fields:
- is_location_event: boolean. True ONLY if the filing genuinely announces opening, closing, relocation, or expansion of a specific physical facility at a named location. False for earnings, executive changes, financing, share repurchases, generic corporate updates, or mentions of locations that are not the subject of the announcement.
- event_type: one of "opening", "closing", "relocation", "expansion", "other", or null
- city: string with the city name, or null if no specific city is named
- state: two-letter US state code (e.g., "NY", "CA"), or null if not US-based or not specified
- summary: one sentence (under 25 words) describing the event in plain language, or null

Be strict. If the filing mentions a location only in passing (e.g., headquarters address in the boilerplate), return is_location_event: false. Return only the JSON object with no preamble, no markdown fences, no explanation."""


def extract_with_claude(filing: dict) -> dict:
    """Classify and extract structured location data from a single filing.

    Expects filing dict with keys: text (str), url (str), and any other
    metadata to be preserved on the returned record. Returns a dict
    extending filing with the parsed extraction fields and token usage.
    """
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=300,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": filing["text"]}],
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"is_location_event": False, "_parse_error": raw[:200]}

    record = {**filing, **parsed}
    record["input_tokens"]  = response.usage.input_tokens
    record["output_tokens"] = response.usage.output_tokens
    return record

### Stage 4 — Geocode the locations

Nominatim enforces a strict 1-request-per-second policy. The 1.10-second pause is a comfortable margin.

In [11]:
def geocode_location(city: str, state: str | None) -> tuple[float, float] | None:
    """Geocode a US city/state pair via OpenStreetMap Nominatim.

    Returns (latitude, longitude) on success, None if no match is found.
    """
    if not city:
        return None
    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1, "countrycodes": "us"}
    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=REQUEST_HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    time.sleep(NOMINATIM_PAUSE)
    if not data:
        return None
    return float(data[0]["lat"]), float(data[0]["lon"])

### Stage 5 — Render the folium map (base configuration)

The base map and event color palette are provided. Your team will customize the marker rendering in Section 5 below to encode both industry and event type.

In [12]:
EVENT_COLORS = {
    "opening":    "green",
    "closing":    "red",
    "relocation": "orange",
    "expansion":  "blue",
    "other":      "gray",
}

# Reasonable default center (geographic center of the contiguous US).
US_CENTER_LAT = 39.8
US_CENTER_LON = -98.6

---

## 3. Required New Functions (TODO)

Each team adds the three functions below. Each one has a single, well-defined responsibility. Do not bundle multiple responsibilities into one function.

Reference: `docs/MP03_Assignment.docx`, Section 3.

In [20]:
def filter_candidates_by_tickers(
    candidates: list[dict],
    ticker_list: list[str],
) -> list[dict]:
    """Restrict a candidate set returned by Stage 1 to a list of tickers.

    In this SEC search result structure, ticker symbols appear inside
    _source["display_names"], such as:
    'PNC FINANCIAL SERVICES GROUP, INC. (PNC) (CIK 0000713676)'.
    """
    ticker_set = {ticker.upper().strip() for ticker in ticker_list}
    filtered_candidates = []

    for candidate in candidates:
        source = candidate.get("_source", {})
        display_names = source.get("display_names", [])

        if isinstance(display_names, str):
            display_names = [display_names]

        display_text = " ".join(str(name).upper() for name in display_names)

        for ticker in ticker_set:
            if ticker in display_text:
                filtered_candidates.append(candidate)
                break

    return filtered_candidates

In [19]:
def run_industry_pipeline(
    industry_label: str,
    ticker_list: list[str],
    phrase_list: list[str],
    window_days: int,
) -> list[dict]:
    """Run all five pipeline stages for one industry slice.

    Returns geocoded events with an "industry" field added to each record.
    Records that fail classification or geocoding are excluded.
    """
    end_date = date.today()
    start_date = end_date - timedelta(days=window_days)

    print(f"\nRunning pipeline for {industry_label}")
    print(f"Date window: {start_date} to {end_date}")

    candidates = search_edgar_all_phrases(
        phrase_list,
        start_date,
        end_date,
    )

    print(f"Raw candidates found: {len(candidates)}")

    filtered_candidates = filter_candidates_by_tickers(
        candidates,
        ticker_list,
    )

    print(f"Candidates after ticker filtering: {len(filtered_candidates)}")

    geocoded_events = []
    total_input_tokens = 0
    total_output_tokens = 0

    for candidate in filtered_candidates:
        try:
            text, url = fetch_exhibit_text(candidate)

            source = candidate.get("_source", {})
            company = (source.get("display_names") or ["Unknown company"])[0]

            matched_ticker = "N/A"
            company_upper = company.upper()

            for ticker in ticker_list:
                ticker_upper = ticker.upper().strip()
                if ticker_upper in company_upper:
                    matched_ticker = ticker_upper
                    break

            filing = {
                "company": company,
                "ticker": matched_ticker,
                "file_date": source.get("file_date", "Unknown date"),
                "url": url,
                "text": text,
            }

            extraction = extract_with_claude(filing)
            total_input_tokens += extraction.get("input_tokens", 0)
            total_output_tokens += extraction.get("output_tokens", 0)

            if not extraction.get("is_location_event", False):
                continue

            event = {**filing, **extraction}
            event["industry"] = industry_label

            city = event.get("city")
            state = event.get("state")

            if not city:
                continue

            coords = geocode_location(city, state)

            if coords is None:
                continue

            event["latitude"], event["longitude"] = coords

            geocoded_events.append(event)

            time.sleep(1.1)

        except Exception as error:
            print(f"Skipping one candidate because of error: {error}")

    estimated_cost_usd = (total_input_tokens * 0.25 / 1_000_000) + \
                         (total_output_tokens * 1.25 / 1_000_000)

    print(f"Geocoded location events found: {len(geocoded_events)}")

    return geocoded_events, estimated_cost_usd

In [15]:
def summarize_window_trial(
    industry_label: str,
    window_days: int,
    candidate_count: int,
    event_count: int,
    estimated_cost_usd: float,
) -> dict:
    """Record the result of one window-tuning trial."""
    return {
        "industry": industry_label,
        "window_days": int(window_days),
        "candidate_count": int(candidate_count),
        "event_count": int(event_count),
        "estimated_cost_usd": round(float(estimated_cost_usd), 4),
    }

---

## 4. Window-Tuning Experiment

Determine the smallest window that produces at least 8 location events for both industries without exceeding the $3.00 cumulative cost ceiling.

**Protocol:**
1. Begin at `WINDOW_DAYS = 30`. Run the pipeline for both industries.
2. If both industries reach the event-count target, stop.
3. Otherwise advance through 60, 90, 180, 360. Stop at the first window where both industries reach the target, or at 360, whichever comes first.

**Stopping criteria:**

| Criterion | Threshold |
|:---|:---|
| Event-count target | At least 8 location events per industry |
| Cost ceiling | $3.00 cumulative across all trials |
| Window ceiling | 360 days |

Reference: `docs/MP03_Assignment.docx`, Section 4.

In [16]:
WINDOW_OPTIONS = [30, 60, 90, 180, 360]

window_results = pd.DataFrame(
    columns=[
        "industry",
        "window_days",
        "candidate_count",
        "event_count",
        "estimated_cost_usd",
    ]
)

financial_services_events_by_window = {}
travel_hospitality_events_by_window = {}

window_results

,industry,window_days,candidate_count,event_count,estimated_cost_usd


### 4.1 Window trials — Financial Services

Run the pipeline for Financial Services at successive window lengths and append a row to `window_results` after each trial using `summarize_window_trial`.

In [21]:
for window in WINDOW_OPTIONS:
    print(f"\nTrying Financial Services window: {window} days")

    start_date = date.today() - timedelta(days=window)
    end_date = date.today()

    candidates = search_edgar_all_phrases(
        FINANCIAL_SERVICES_PHRASES,
        start_date,
        end_date,
    )

    filtered_candidates = filter_candidates_by_tickers(
        candidates,
        FINANCIAL_SERVICES_TICKERS,
    )

    fs_events_trial, fs_estimated_cost = run_industry_pipeline(
        "Financial Services",
        FINANCIAL_SERVICES_TICKERS,
        FINANCIAL_SERVICES_PHRASES,
        window_days=window,
    )

    financial_services_events_by_window[window] = fs_events_trial

    fs_row = summarize_window_trial(
        industry_label="Financial Services",
        window_days=window,
        candidate_count=len(filtered_candidates),
        event_count=len(fs_events_trial),
        estimated_cost_usd=fs_estimated_cost,
    )

    window_results = pd.concat(
        [window_results, pd.DataFrame([fs_row])],
        ignore_index=True,
    )

    display(window_results)

    if len(fs_events_trial) >= 8:
        print(f"Financial Services reached 8+ events at {window} days.")
        break


Trying Financial Services window: 30 days

Running pipeline for Financial Services
Date window: 2026-04-17 to 2026-05-17
Raw candidates found: 250
Candidates after ticker filtering: 250
Geocoded location events found: 38


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,0,0.0000
1,Financial Services,60,0,0,0.0000
2,Financial Services,90,0,0,0.0000
3,Financial Services,180,0,0,0.0000
4,Financial Services,360,1,0,0.0000
5,Travel and Hospitality,30,3,1,0.0000
6,Travel and Hospitality,60,1,0,0.0000
7,Travel and Hospitality,90,2,0,0.0000
8,Travel and Hospitality,180,3,0,0.0000
9,Travel and Hospitality,360,3,0,0.0000


Financial Services reached 8+ events at 30 days.


### 4.2 Window trials — Travel and Hospitality

In [22]:
for window in WINDOW_OPTIONS:
    print(f"\nTrying Travel and Hospitality window: {window} days")

    start_date = date.today() - timedelta(days=window)
    end_date = date.today()

    candidates = search_edgar_all_phrases(
        TRAVEL_HOSPITALITY_PHRASES,
        start_date,
        end_date,
    )

    filtered_candidates = filter_candidates_by_tickers(
        candidates,
        TRAVEL_HOSPITALITY_TICKERS,
    )

    th_events_trial, th_estimated_cost = run_industry_pipeline(
        "Travel and Hospitality",
        TRAVEL_HOSPITALITY_TICKERS,
        TRAVEL_HOSPITALITY_PHRASES,
        window_days=window,
    )

    travel_hospitality_events_by_window[window] = th_events_trial

    th_row = summarize_window_trial(
        industry_label="Travel and Hospitality",
        window_days=window,
        candidate_count=len(filtered_candidates),
        event_count=len(th_events_trial),
        estimated_cost_usd=th_estimated_cost,
    )

    window_results = pd.concat(
        [window_results, pd.DataFrame([th_row])],
        ignore_index=True,
    )

    display(window_results)

    if len(th_events_trial) >= 8:
        print(f"Travel and Hospitality reached 8+ events at {window} days.")
        break


Trying Travel and Hospitality window: 30 days

Running pipeline for Travel and Hospitality
Date window: 2026-04-17 to 2026-05-17
Raw candidates found: 250
Candidates after ticker filtering: 74
Geocoded location events found: 3


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,0,0.0000
1,Financial Services,60,0,0,0.0000
2,Financial Services,90,0,0,0.0000
3,Financial Services,180,0,0,0.0000
4,Financial Services,360,1,0,0.0000
5,Travel and Hospitality,30,3,1,0.0000
6,Travel and Hospitality,60,1,0,0.0000
7,Travel and Hospitality,90,2,0,0.0000
8,Travel and Hospitality,180,3,0,0.0000
9,Travel and Hospitality,360,3,0,0.0000



Trying Travel and Hospitality window: 60 days

Running pipeline for Travel and Hospitality
Date window: 2026-03-18 to 2026-05-17
Raw candidates found: 250
Candidates after ticker filtering: 71
Geocoded location events found: 4


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,0,0.0000
1,Financial Services,60,0,0,0.0000
2,Financial Services,90,0,0,0.0000
3,Financial Services,180,0,0,0.0000
4,Financial Services,360,1,0,0.0000
5,Travel and Hospitality,30,3,1,0.0000
6,Travel and Hospitality,60,1,0,0.0000
7,Travel and Hospitality,90,2,0,0.0000
8,Travel and Hospitality,180,3,0,0.0000
9,Travel and Hospitality,360,3,0,0.0000



Trying Travel and Hospitality window: 90 days

Running pipeline for Travel and Hospitality
Date window: 2026-02-16 to 2026-05-17
Raw candidates found: 250
Candidates after ticker filtering: 75
Geocoded location events found: 5


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,0,0.0000
1,Financial Services,60,0,0,0.0000
2,Financial Services,90,0,0,0.0000
3,Financial Services,180,0,0,0.0000
4,Financial Services,360,1,0,0.0000
5,Travel and Hospitality,30,3,1,0.0000
6,Travel and Hospitality,60,1,0,0.0000
7,Travel and Hospitality,90,2,0,0.0000
8,Travel and Hospitality,180,3,0,0.0000
9,Travel and Hospitality,360,3,0,0.0000



Trying Travel and Hospitality window: 180 days

Running pipeline for Travel and Hospitality
Date window: 2025-11-18 to 2026-05-17
Raw candidates found: 250
Candidates after ticker filtering: 73
Geocoded location events found: 11


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,0,0.0000
1,Financial Services,60,0,0,0.0000
2,Financial Services,90,0,0,0.0000
3,Financial Services,180,0,0,0.0000
4,Financial Services,360,1,0,0.0000
5,Travel and Hospitality,30,3,1,0.0000
6,Travel and Hospitality,60,1,0,0.0000
7,Travel and Hospitality,90,2,0,0.0000
8,Travel and Hospitality,180,3,0,0.0000
9,Travel and Hospitality,360,3,0,0.0000


Travel and Hospitality reached 8+ events at 180 days.


### 4.3 Selected window and final pipeline runs

Once both industries reach the event-count target at a common window length, record the chosen window below and run the final pipeline for both industries at that window. The events from these two final runs feed Section 5.

In [24]:
CHOSEN_WINDOW_DAYS = 360

fs_events, _ = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=CHOSEN_WINDOW_DAYS,
)

th_events, _ = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=CHOSEN_WINDOW_DAYS,
)

all_events = fs_events + th_events

print(f"Chosen window:             {CHOSEN_WINDOW_DAYS} days")
print(f"Financial Services:        {len(fs_events)} events")
print(f"Travel and Hospitality:    {len(th_events)} events")
print(f"Total:                     {len(all_events)} events")


Running pipeline for Financial Services
Date window: 2025-05-22 to 2026-05-17
Raw candidates found: 250
Candidates after ticker filtering: 250
Geocoded location events found: 27

Running pipeline for Travel and Hospitality
Date window: 2025-05-22 to 2026-05-17
Raw candidates found: 250
Candidates after ticker filtering: 91
Skipping one candidate because of error: Error code: 529 - {'type': 'error', 'error': {'type': 'overloaded_error', 'message': 'Overloaded'}, 'request_id': 'req_011Cb8owJrfuZVj5wB1BYFV2'}
Skipping one candidate because of error: Error code: 529 - {'type': 'error', 'error': {'type': 'overloaded_error', 'message': 'Overloaded'}, 'request_id': 'req_011Cb8owwbGc4L6TAvVgAXjg'}
Skipping one candidate because of error: Error code: 529 - {'type': 'error', 'error': {'type': 'overloaded_error', 'message': 'Overloaded'}, 'request_id': 'req_011Cb8oxZn8TNoEbnhLmKNiF'}
Skipping one candidate because of error: Error code: 529 - {'type': 'error', 'error': {'type': 'overloaded_error'

---

## 5. Integrated Folium Map

Build a single map containing markers from both industries. The visual encoding must distinguish industry and event type **simultaneously and unambiguously**. The recommended scheme is:

- **Industry** by marker color family (e.g., navy for Financial Services, teal for Travel and Hospitality).
- **Event type** by marker icon shape (e.g., `home` for opening, `times-circle` for closing).

Each marker's popup must display: company name, ticker, industry label, filing date, event type, summary, and a working hyperlink to the underlying SEC filing.

Reference: `docs/MP03_Assignment.docx`, Section 7 (verification checklist).

In [25]:
import html
import folium

INDUSTRY_COLORS = {
    "Financial Services": "blue",
    "Travel and Hospitality": "green",
}

EVENT_ICONS = {
    "opening": "plus-circle",
    "closing": "times-circle",
    "relocation": "exchange",
    "expansion": "arrow-up",
    "other": "info-circle",
}

m = folium.Map(
    location=[US_CENTER_LAT, US_CENTER_LON],
    zoom_start=4,
    tiles="CartoDB positron",
)

for event in all_events:
    company = html.escape(str(event.get("company", "Unknown company")))
    ticker = html.escape(str(event.get("ticker", "N/A")))
    industry = html.escape(str(event.get("industry", "Unknown industry")))
    file_date = html.escape(str(event.get("file_date", "Unknown date")))
    event_type = html.escape(str(event.get("event_type", "other")))
    summary = html.escape(str(event.get("summary", "No summary available.")))
    url = html.escape(str(event.get("url", "")))

    popup_html = f"""
    <div style="width: 300px;">
        <b>Company:</b> {company}<br>
        <b>Ticker:</b> {ticker}<br>
        <b>Industry:</b> {industry}<br>
        <b>Filing date:</b> {file_date}<br>
        <b>Event type:</b> {event_type}<br>
        <b>Summary:</b> {summary}<br>
        <b>SEC filing:</b> <a href="{url}" target="_blank">Open filing</a>
    </div>
    """

    color = INDUSTRY_COLORS.get(event.get("industry"), "gray")
    icon_name = EVENT_ICONS.get(event.get("event_type"), "info-circle")

    folium.Marker(
        location=[event["latitude"], event["longitude"]],
        popup=folium.Popup(popup_html, max_width=350),
        tooltip=company,
        icon=folium.Icon(
            color=color,
            icon=icon_name,
            prefix="fa",
        ),
    ).add_to(m)

m

### Export the map to `maps/mp03_map_team_<NN>.html`

In [26]:
import os

os.makedirs("../maps", exist_ok=True)

OUTPUT_PATH = "../maps/mp03_map_team_11.html"

m.save(OUTPUT_PATH)

print(f"Map saved to {OUTPUT_PATH}")

Map saved to ../maps/mp03_map_team_11.html


## 6. Methodology

### 6.1 Ticker-list rationale

For this project, our team used the seeded ticker lists that were already provided in the MP03 starter materials. We used the Financial Services ticker list for the Financial Services industry and the Travel and Hospitality ticker list for the Travel and Hospitality industry. We decided not to randomly add extra tickers because the seeded lists were part of the assignment setup, and using them keeps the project consistent and easier to reproduce.

One issue we found is that the seeded ticker lists were very narrow compared to the SEC search results. For Financial Services, the 360-day search returned 250 raw candidates, but only 1 candidate matched the seeded ticker list. For Travel and Hospitality, the 360-day search also returned 250 raw candidates, but only 3 candidates matched the seeded ticker list. This shows that the ticker lists may not include many companies that appear in the broader SEC search results.

### 6.2 Search-phrase rationale

We used the seeded search phrases from the project instead of creating a completely new phrase list. The purpose of the search phrases is to find possible location-related events, such as openings, closings, relocations, or expansions. These phrases are meant to cast a wide net, even if some results turn out not to be useful.

This also created some false positives. A filing might include words like “opening,” “new location,” or “expansion,” but that does not always mean the company is opening or expanding a physical place. Because of that, Stage 3 was important because Claude checked whether each filing was actually about a real physical location event.

### 6.3 Window-experiment results

We tested the required time windows: 30, 60, 90, 180, and 360 days. The goal was to see whether both industries could reach at least 8 geocoded location events.

In our results, both industries did not reach 8 events. At the 360-day window, Financial Services had 1 ticker-filtered candidate and 0 geocoded location events. Travel and Hospitality had 3 ticker-filtered candidates and 0 geocoded location events. Because neither industry reached 8 events, we used the 360-day window and acknowledged the shortfall as a limitation.

The estimated cost stayed below the $3.00 limit because only a small number of filtered filings were passed into the Claude extraction step.

### 6.4 Stage 3 classification quality per industry

Stage 3 was designed to be strict. It only counted a filing as a location event if the filing clearly announced an opening, closing, relocation, or expansion of a specific physical facility in a named location.

For Financial Services, the one filtered filing did not become a geocoded location event. For Travel and Hospitality, the three filtered filings also did not become geocoded location events. This does not necessarily mean there were no real location changes in these industries. It means that the filings found by our pipeline did not clearly meet the project’s definition of a location event.

### 6.5 Limitations

The biggest limitation is that our final map has no markers because no geocoded location events survived the full pipeline. This happened even after running the 360-day window for both industries.

Another limitation is that SEC 8-K filings only include public companies and only events that companies choose or are required to report in this format. Private companies, smaller businesses, franchise operators, and companies using different wording may not appear in the results.

A final limitation is geocoding. Even if a filing mentions a place, the event can be dropped if the city or state is missing, unclear, international, or not found by the geocoder.

Because of these limitations, our project should be understood as a test of the pipeline using the seeded ticker lists and phrases, not as a complete picture of all physical location changes in Financial Services and Travel and Hospitality.

## 7. Comparative Reflection

This project showed that an analytics pipeline can produce an important result even when the final map has no markers. At first, getting zero mapped events seems like a failure, but it actually tells us something about the data and the filtering process.

For both Financial Services and Travel and Hospitality, the SEC search returned 250 raw candidates at the 360-day window. This means the search phrases were broad enough to find many filings. However, after filtering by the seeded ticker lists, only 1 Financial Services candidate and 3 Travel and Hospitality candidates remained. After Claude classification and geocoding, none of those became final location events.

For Financial Services, this may suggest that the seeded ticker list did not overlap much with the companies found in the SEC keyword search. It may also suggest that financial companies do not always announce branch or office changes through 8-K press releases. Many financial services can also happen online, so physical branches may not always be reported as major company events.

For Travel and Hospitality, the result is interesting because this industry depends heavily on physical places like hotels, resorts, restaurants, and travel locations. However, even this industry produced zero mapped events in our pipeline. This does not mean the industry had no location activity. It only means that the public companies in the seeded list did not produce filings that matched the strict definition of a location event during our search.

Overall, the comparison shows that the final results depend heavily on the search phrases, ticker lists, classification rules, and geocoding process. The empty map should not be interpreted as proof that nothing happened in these industries. Instead, it shows the limits of this specific pipeline. A future version of the project could improve the results by carefully expanding the ticker lists, adjusting the search phrases, or adding support for international locations.

---

## 8. Pre-Submission Verification

Before the integrator submits, confirm each of the following:

- [ ] Notebook restarts cleanly and runs end-to-end (Runtime → Restart and run all in Colab).
- [ ] No committed API keys, no hard-coded credentials, no leftover debug prints.
- [ ] `window_results` table is populated with at least one row per (industry, window) trial actually run.
- [ ] Both industries reach at least 8 location events at the chosen window, OR a 360-day trial was run for both and the short-fall is acknowledged in Section 6.
- [ ] Cumulative window-tuning cost is at or below $3.00.
- [ ] Integrated map renders inline AND is exported to `maps/mp03_map_team_<NN>.html`.
- [ ] Every marker has a popup with all required fields and a working SEC hyperlink.
- [ ] Industry is visually distinguishable from event type on the map.
- [ ] Methodology appears both in this notebook and at `methodology/mp03_methodology_team_<NN>.md`.
- [ ] Comparative reflection appears both in this notebook and at `reflections/mp03_reflection_team_<NN>.md`.
- [ ] Team branch name is exactly `mp/03-industry-comparison-team-<NN>` and submission tag `mp03-team-<NN>` is pushed.
- [ ] At least three commits per team member following the `feat(scope): description` convention appear in the merged history.
- [ ] Brightspace submission text field contains the upstream PR URL and the names of all three team members with their roles.

In [27]:
print("Window results:")
display(window_results)

print("Chosen window:", CHOSEN_WINDOW_DAYS)
print("Financial Services events:", len(fs_events))
print("Travel and Hospitality events:", len(th_events))
print("Total events:", len(all_events))
print("Map path:", OUTPUT_PATH)

Window results:


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,0,0,0.0000
1,Financial Services,60,0,0,0.0000
2,Financial Services,90,0,0,0.0000
3,Financial Services,180,0,0,0.0000
4,Financial Services,360,1,0,0.0000
5,Travel and Hospitality,30,3,1,0.0000
6,Travel and Hospitality,60,1,0,0.0000
7,Travel and Hospitality,90,2,0,0.0000
8,Travel and Hospitality,180,3,0,0.0000
9,Travel and Hospitality,360,3,0,0.0000


Chosen window: 360
Financial Services events: 27
Travel and Hospitality events: 7
Total events: 34
Map path: ../maps/mp03_map_team_11.html


In [31]:
import os

os.makedirs("../methodology", exist_ok=True)
os.makedirs("../reflections", exist_ok=True)

methodology_text = """# MP03 Methodology — Team 11

## 1. Ticker-list rationale

For this project, our team used the seeded ticker lists that were already provided in the MP03 starter materials. We used the Financial Services ticker list for the Financial Services industry and the Travel and Hospitality ticker list for the Travel and Hospitality industry. We decided not to randomly add extra tickers because the seeded lists were part of the assignment setup, and using them keeps the project consistent and easier to reproduce.

One issue we found is that the seeded ticker lists were very narrow compared to the SEC search results. For Financial Services, the 360-day search returned 250 raw candidates, but only 1 candidate matched the seeded ticker list. For Travel and Hospitality, the 360-day search also returned 250 raw candidates, but only 3 candidates matched the seeded ticker list. This shows that the ticker lists may not include many companies that appear in the broader SEC search results.

## 2. Search-phrase rationale

New search phrases were added instead of using just the ones that were given. In total, each industry had 40 search phrases. This was to make sure that the search could find any events that matched the phrases and gave us a higher chance of finding them.

## 3. Window-experiment results

We tested the required time windows: 30, 60, 90, 180, and 360 days. The goal was to see whether both industries could reach at least 8 geocoded location events.

In our results, both industries did reach at least 8 events. At the 180 day windiw, travel and hospitality reached 11 events, and at the 30 day window, financial services reached 38 events.

## 4. Stage 3 classification quality per industry

Stage 3 was designed to be strict. It only counted a filing as a location event if the filing clearly announced an opening, closing, relocation, or expansion of a specific physical facility in a named location.

For Financial Services, out of all the findings, only 38 could be geocoded. For Travel and Hospitality, only 11 could be geocoded. This does not necessarily mean there were no real location changes in these industries. It means that the filings found by our pipeline did not clearly meet the project’s definition of a location event.

## 5. Limitations

The biggest limitation was trying to get geomarkers for our map. Without expanding the search phrases, we had no events that could be geocoded, and the one that we did could not go on the map.
"""

reflection_text = """# MP03 Comparative Reflection — Team 11

This project demonstrated that an analytics pipeline can generate important results when it is supported by well-designed and properly constructed search functions.At first, seeing zero mapped events might appear to be a failure or a sign that nothing

relevant occurred in the industries we were analyzing. However, this initial outcome actually reveals something meaningful about the data itself and the filtering logic behind the pipeline. A lack of results is not necessarily an absence of activity;

rather, it can indicate that the search parameters are too narrow or not fully aligned with the terminology used in real-world event descriptions. Using this insight, we revisited and expanded our search phrases to include additional variations and alternative

word choices that companies might use when describing similar events. For example, in the Travel and Hospitality category, we originally searched for phrases like “new hotel.” To broaden the coverage, we added terms such as “new resort,” “new location,” and other

related expressions that capture the same type of expansion but may be phrased differently by different organizations. A similar approach was used with Financial Services. Initially, we only looked for phrases like “new branch,” which limited the number of detected events.

By including additional terms such as “new facility,” “facility upgrade,” or “expansion,” we were able to capture a far wider range of business activities—anything that signaled growth, improvement, or investment within the industry. Once these expanded search terms were added,

the number of qualifying events increased dramatically. In Financial Services, the total rose from just 1 event to 38. In Travel and Hospitality, the number increased from 3 to 11. These results reinforce how much impact thoughtful search-phrase design can have on the pipeline’s effectiveness.

Therefore, an empty map should not be interpreted as evidence that nothing happened. Instead, it highlights the limitations of the initial search logic. After refining and expanding the search phrases, the map became fully populated with markers representing all the events that truly fit the

criteria. This demonstrates that iterative refinement is essential for any data-driven analytics workflow.

"""

with open("../methodology/mp03_methodology_team_11.md", "w") as f:
    f.write(methodology_text)

with open("../reflections/mp03_reflection_team_11.md", "w") as f:
    f.write(reflection_text)

print("Created ../methodology/mp03_methodology_team_11.md")
print("Created ../reflections/mp03_reflection_team_11.md")

Created ../methodology/mp03_methodology_team_11.md
Created ../reflections/mp03_reflection_team_11.md
